# Departments Exploratory Data Analysis

## Purpose

This notebook checks department data. It checks missing values, IDs, dates, and links to organisations and employees.

## Files used

- `Departments.csv` — department records
- `Organisations.csv` — organisation records
- `Employees.csv` — employees linked to departments
- `Branches.csv` — branch records

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (directory / 'data' / 'raw', directory / 'data-analytics' / 'data' / 'raw'):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError('Could not locate data-analytics/data/raw')

RAW_DATA_DIR = find_raw_data_dir()

## 1. Load the data

In [ ]:
departments = pd.read_csv(RAW_DATA_DIR / 'Departments.csv')
organisations = pd.read_csv(RAW_DATA_DIR / 'Organisations.csv')
employees = pd.read_csv(RAW_DATA_DIR / 'Employees.csv')
branches = pd.read_csv(RAW_DATA_DIR / 'Branches.csv')
pd.DataFrame({
    'dataset': ['Departments', 'Organisations', 'Employees', 'Branches'],
    'rows': [len(departments), len(organisations), len(employees), len(branches)],
    'columns': [len(departments.columns), len(organisations.columns), len(employees.columns), len(branches.columns)],
})

## 2. Check the department file

In [ ]:
departments

In [ ]:
department_profile = pd.DataFrame({
    'data_type': departments.dtypes.astype(str),
    'missing_count': departments.isna().sum(),
    'unique_values': departments.nunique(dropna=False),
})
department_profile

In [ ]:
required_columns = ['department_id', 'organisation_id', 'name', 'created_at']
assert set(required_columns).issubset(departments.columns)
assert departments[required_columns].notna().all().all()
print('All required department fields have values.')

## 3. Check IDs, names, and dates

In [ ]:
department_dates = pd.to_datetime(departments['created_at'], utc=True, errors='coerce')
quality_checks = pd.Series({
    'duplicate_department_ids': departments['department_id'].duplicated().sum(),
    'wrong_department_id_format': (~departments['department_id'].str.match(r'^DEP-[0-9]{3}$', na=False)).sum(),
    'wrong_organisation_id_format': (~departments['organisation_id'].str.match(r'^ORG-[0-9]{3}$', na=False)).sum(),
    'duplicate_names_in_organisation': departments.duplicated(['organisation_id', 'name']).sum(),
    'names_with_extra_spaces': departments['name'].ne(departments['name'].str.strip()).sum(),
    'bad_created_dates': department_dates.isna().sum(),
    'exact_duplicate_rows': departments.duplicated().sum(),
})
quality_checks.to_frame('count')

In [ ]:
assert quality_checks.eq(0).all()
print('Department IDs, names, and dates are valid.')

## 4. Check organisation and employee links

In [ ]:
unknown_organisation_ids = sorted(set(departments['organisation_id']) - set(organisations['organisation_id']))
unknown_employee_department_ids = sorted(set(employees['department_id']) - set(departments['department_id']))
departments_without_employees = sorted(set(departments['department_id']) - set(employees['department_id']))
employee_department_join = employees.merge(
    departments[['department_id', 'name', 'organisation_id']].rename(columns={'name': 'department_name', 'organisation_id': 'department_organisation_id'}),
    on='department_id', how='left', validate='many_to_one',
)
organisation_mismatches = employee_department_join['organisation_id'].ne(employee_department_join['department_organisation_id']).sum()
link_checks = pd.Series({
    'unknown_organisation_ids': len(unknown_organisation_ids),
    'unknown_employee_department_ids': len(unknown_employee_department_ids),
    'departments_without_employees': len(departments_without_employees),
    'employee_department_organisation_mismatches': organisation_mismatches,
})
link_checks.to_frame('count')

In [ ]:
assert link_checks.eq(0).all()
print('All departments and employees link to the right organisation.')

## 5. Count employees by department and branch

In [ ]:
employees_by_department = employee_department_join.groupby(['department_id', 'department_name']).size().rename('employee_count').reset_index()
department_branch_table = pd.crosstab(employee_department_join['department_name'], employee_department_join['branch_id'])
display(employees_by_department)
display(department_branch_table)

In [ ]:
ax = employees_by_department.plot(kind='barh', x='department_name', y='employee_count', legend=False, color='#176B87', figsize=(8, 4), title='Employees by department')
ax.set_xlabel('Employee count')
ax.set_ylabel('')
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 6. Findings

There are 4 departments. All required fields have values. Department IDs are unique and correctly formatted. Department names are unique inside the organisation. All dates are valid.

All departments link to the known organisation. All 28 employees link to known departments in the same organisation. No department is empty.

Each department has 7 employees. Every department has employees in both branches. The data is evenly split by department, but the dataset has only one organisation.

## Next steps

- Keep the ID, name, date, and relationship checks.
- Add more organisations before comparing department structures.
- Review department sizes again when more employees are added.
- Confirm that departments are shared across branches.